In [ ]:
# Analyzing how different factors such as house size, house loaction determine the price of houses in the real estate market in Brazil

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [ ]:
# Import first csv file
df1 = pd.read_csv("/Users/nxcho/Desktop/Data Science/housing_in_Brazil/data/raw/brasil_real_estate_1.csv")
df1.head()

In [ ]:
def clean_df1(df):
    """This function drops NaN values from a dataframe,
    creates a "state", "city", "lat" and "lon" column,
    coverts the price string into a float and
    drops junk columns"""

    # Dataset copy
    df = df.copy()
    
    # Dropping NaN (missing values)
    df.dropna(inplace = True, ignore_index = True)
    assert df.isna().sum().sum() == 0, "unexpected NaNs remain after cleaning"

    # Creating a "state" column from "place_with_parent_names"
    df["state"] = df["place_with_parent_names"].str.split("|", expand = True)[2]
    assert df["state"].notna().all(), "state extraction failed for some rows"

    # Creating a "city" column from "place_with_parent_names"
    df["city"] = df["place_with_parent_names"].str.split("|", expand = True)[3]
    assert df["city"].notna().all(), "city extraction failed for some rows"

    # Creating a "lat" column from "lat-lon"
    df["lat"] = df["lat-lon"].str.split(",", expand = True)[0].astype(float)
    assert df["lat"].dtype == float, "lat should be numeric"

    # Creating a "lon" column from "lat-lon"
    df["lon"] = df["lat-lon"].str.split(",", expand = True)[1].astype(float)
    assert df["lon"].dtype == float, "lon should be numeric"

    # Converting "price_usd" from string to float
    df["price_usd"] = df["price_usd"].str.replace("$", "", regex = False).str.replace(",", "").astype(float)
    assert df["price_usd"].dtype == float, "price_usd should be float after cleaning"

    # Dropping the duplicate columns
    df.drop(columns = ["place_with_parent_names", "lat-lon"], inplace = True)
    
    return df

In [ ]:
# Cleaning df1
df1 = clean_df1(df1)

In [ ]:
# Importing second csv file
df2 = pd.read_csv("/Users/nxcho/Desktop/Data Science/housing_in_Brazil/data/raw/brasil_real_estate_2.csv")
df2.head()

In [ ]:
def clean_df2(df):
    """This function drops NaN values from a dataframe,
    creates a "state", "city", "lat" and "lon" column,
    coverts the "price_brl" column to USD and
    drops junk columns"""

    # Dataset copy
    df = df.copy()
    
    # Dropping NaN (missing values)
    df.dropna(inplace = True, ignore_index = True)
    assert df.isna().sum().sum() == 0, "unexpected NaNs remain after cleaning"

    # Creating a "state" column from "place_with_parent_names"
    df["state"] = df["place_with_parent_names"].str.split("|", expand = True)[2]
    assert df["state"].notna().all(), "state extraction failed for some rows"

    # Creating a "city" column from "place_with_parent_names"
    df["city"] = df["place_with_parent_names"].str.split("|", expand = True)[3]
    assert df["city"].notna().all(), "city extraction failed for some rows"

    # Creating a "lat" column from "lat-lon"
    df["lat"] = df["lat-lon"].str.split(",", expand = True)[0].astype(float)
    assert df["lat"].dtype == float, "lat should be numeric"

    # Creating a "lon" column from "lat-lon"
    df["lon"] = df["lat-lon"].str.split(",", expand = True)[1].astype(float)
    assert df["lon"].dtype == float, "lon should be numeric"

    # Converting "price_brl" from string to "price_usd"
    df["price_usd"] = df["price_brl"].astype(float)/3.19
    assert df["price_usd"].dtype == float, "price_usd should be float after cleaning"

    # Dropping the duplicate columns
    df.drop(columns = ["place_with_parent_names", "lat-lon", "price_brl"], inplace = True)
    
    return df

In [ ]:
# Cleaning df2
df2 = clean_df2(df2)

In [ ]:
# Combining df1 and df2
df = pd.concat([df1, df2], ignore_index=True)
assert set(df.columns) == {"property_type", "region", "area_m2", "price_usd", "state", "city", "lat", "lon"}
assert df.isna().sum().sum() == 0

In [ ]:
# Creating a scatter map centered on Brazil 
fig = px.scatter_map(
    df,
    lat = "lat",
    lon = "lon",
    center = {"lat": -14.2, "lon": -51.9}, # Centered on Brazil
    width = 600,
    height = 600,
    hover_data = ["price_usd"],
)

fig.update_layout(map_style = "open-street-map")

fig.show()

In [ ]:
# Histogram of frequency of "price_usd"
price_hist = plt.hist(
    df["price_usd"]
)

plt.xlabel("Price [USD]")
plt.ylabel("Frequency")
plt.title("Distribution of house prices")

plt.show()

In [ ]:
# Boxplot to identify outliers
plt.boxplot(
    df["area_m2"],
    orientation = "horizontal"
)

plt.xlabel("Area [sq meters]")
plt.title("Distribution of home size")

plt.show()

In [ ]:
# Mean price by region
mean_price_by_region = df.groupby("region")["price_usd"].mean().sort_values(ascending = True)
mean_price_by_region

In [ ]:
# Bar graph of mean price by region
mean_price_by_region.plot(
    kind = "bar",
    xlabel = "Region",
    ylabel = "Price [USD]"
)

plt.title("A bar graph showing the mean price by region")

plt.show()

In [ ]:
# Correlation between "area_m2" and "price_usd" by region
region_corr = df.groupby("region")[["area_m2", "price_usd"]].apply(
    lambda g: g["area_m2"].corr(g["price_usd"])
).sort_values()

sns.heatmap(
    region_corr.to_frame(),
    annot = True,
    cmap = "coolwarm",
    vmin = -1,
    vmax = 1,
    fmt = ".2f"
)

plt.title("Area–Price Correlation by Region")

plt.show()

In [ ]:
# Histograms of price distribution by region
regions = sorted(df['region'].unique())
fig, axes = plt.subplots(len(regions), 1, figsize=(8, 4 * len(regions)))

for i, region in enumerate(regions):
    subset = df[df['region'] == region]
    axes[i].hist(subset['price_usd'], bins=30, color='skyblue', edgecolor='black')
    axes[i].set_title(f'Price Distribution in {region}')
    axes[i].set_xlabel('Price [USD]')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Variance in price by region
df.groupby("region")["price_usd"].var()

In [ ]:
# Kruskal-Wallis test across regions
groups = [g["price_usd"].values for _, g in df.groupby("region")]
stat, p_value = kruskal(*groups)
print(f"{p_value:.2e}")

In [ ]:
# Run Dunn's Test
dunn_results = sp.posthoc_dunn(
    df,
    val_col = "price_usd",
    group_col = "region",
    p_adjust = "holm"
)

print(dunn_results)

In [ ]:
# Compute Epsilon-Squared
total_n = len(df)
epsilon_sq = stat / (total_n - 1)

print(f"Kruskal-Wallis H: {stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"Epsilon-Squared: {epsilon_sq:.4f}")

In [ ]:
# Train_test_split
X = df[["area_m2"]]
y = df["price_usd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
# Naive baseline prediction on training data
baseline_pred_test = y_train.median()
mae_baseline_test = mean_absolute_error(y_test, [baseline_pred_test] * len(y_test))
print(mae_baseline_test)

In [ ]:
# Linear Regression model
lr = LinearRegression()
lr.fit(X_train, y_train)
print("Slope:", lr.coef_[0])
print("Intercept:", lr.intercept_)

In [ ]:
# Train Linear Regression model
y_pred = lr.predict(X_test)

In [ ]:
# Model evaluation
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", round(mae, 2))

In [ ]:
lr.score(X_test, y_test)